In [16]:
pip install konlpy



   ---------------------------------------- 0.0/19.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.4 MB 330.3 kB/s eta 0:00:59
   ---------------------------------------- 0.1/19.4 MB 737.3 kB/s eta 0:00:27
    --------------------------------------- 0.4/19.4 MB 2.8 MB/s eta 0:00:07
   - -------------------------------------- 0.8/19.4 MB 3.8 MB/s eta 0:00:05
   -- ------------------------------------- 1.2/19.4 MB 4.6 MB/s eta 0:00:04
   -- ------------------------------------- 1.3/19.4 MB 4.2 MB/s eta 0:00:05
   --- ------------------------------------ 1.7/19.4 MB 4.7 MB/s eta 0:00:04
   ---- ----------------------------------- 2.1/19.4 MB 5.4 MB/s eta 0:00:04
   ----- ---------------------------------- 2.5/19.4 MB 5.8 MB/s eta 0:00:03
   ------ ------------

In [23]:
# -*- coding: utf-8 -*-
import pandas as pd
import re
import chardet
import os, time

INPUT_FILE = r"C:/Users/alstj/Documents/카카오톡 받은 파일/레시피_날것_재료목록_클린.csv"
BASE_OUTPUT = r"C:/Users/alstj/Desktop/백종원레시피정제_final_stable_v3.csv"

# 열려있어도 저장되게 파일명 자동 변경
OUTPUT_FILE = BASE_OUTPUT if not os.path.exists(BASE_OUTPUT) else BASE_OUTPUT.replace(
    ".csv", f"_{time.strftime('%Y%m%d_%H%M%S')}.csv"
)

# =========================
# 단위 표준화
# =========================
UNIT_MAP = {
    "g": "g", "그램": "g",
    "kg": "kg",
    "ml": "ml",
    "l": "l", "L": "l",
    "컵": "cup",
    "큰술": "tbsp",
    "작은술": "tsp",
    "개": "piece"
}

UNIT_CONVERSION = {
    "g": 1, "kg": 1000,
    "ml": 1, "l": 1000,
    "tbsp": 15,
    "tsp": 5,
    "cup": 200,
    "piece": 1
}

APPROX_PATTERN = r"(한\s?줌|한\s?주먹|꼬집|한\s?꼬집|약간|적당히|적당량|조금|약\s?\d+|듬뿍|취향껏)"

# 조리/설명 문장 걸러내기 강화
COOKING_VERB_PATTERN = r"(넣|끓|볶|굽|자르|섞|씻|데치|익|빼|만들|담|조리)"
POLITE_ENDING_PATTERN = r"(습니다|니다|세요|해요|했어요|하겠습니다|바랍니다|입니다|됩니다|없습니다|없으니|있습니다)"
EXPLAIN_KEYWORDS_PATTERN = r"(영상|레시피|공개|다녀왔|인체|해가|팀원|여러분|바랍니다|소개|설명|오늘|다음|좋은\s?음식|드시면)"

# =========================
# 유틸
# =========================
def detect_encoding(path: str) -> str:
    with open(path, 'rb') as f:
        raw = f.read()
    return chardet.detect(raw).get("encoding") or "utf-8"

def clean_text(text) -> str:
    text = str(text)
    text = re.sub(r"\(.*?\)", "", text)   # 괄호 제거
    text = re.sub(r"\*", "", text)        # * 제거
    return text.strip()

def is_english_only(text: str) -> bool:
    return re.match(r'^[A-Za-z\s]+$', text) is not None

def looks_like_list_index(text: str) -> bool:
    t = text.strip()
    m = re.match(r'^(\d{1,2})\s+', t)
    if not m:
        return False
    idx = int(m.group(1))
    return 1 <= idx <= 50

def is_instruction_or_explain(text: str) -> bool:
    """
    재료가 아닌 '설명/문장/멘트/조리과정' 제거
    """
    t = text.strip()

    if t == "":
        return True

    # 너무 김 = 문장일 확률 높음
    if len(t) >= 35:
        return True

    # 동사/조리 동작
    if re.search(COOKING_VERB_PATTERN, t):
        return True

    # 존댓말/문장 종결
    if re.search(POLITE_ENDING_PATTERN, t):
        return True

    # 설명 키워드
    if re.search(EXPLAIN_KEYWORDS_PATTERN, t):
        return True

    # 전형적 문장 종결(다/요 포함)
    if t.endswith(("다", "다.", "요", "요.")):
        return True

    # 조사/접속 포함 + 길이 길면 문장 가능성 높음
    if len(t) >= 18 and re.search(r"(은|는|이|가|을|를|에|에서|으로|와|과)\b", t):
        return True

    return False

def convert_fraction(q: str) -> float:
    if "/" in q:
        num, denom = q.split("/")
        return float(num) / float(denom)
    return float(q)

def normalize_unit(unit: str):
    return UNIT_MAP.get(unit, None)

def convert_to_base(quantity, unit_code):
    if quantity is None or unit_code is None:
        return None
    return quantity * UNIT_CONVERSION.get(unit_code, 1)

def move_trailing_unit_to_unitcode(name: str, unit_code):
    if unit_code is not None:
        return name, unit_code

    n = name.strip()

    for u in ["큰술", "작은술", "컵", "개"]:
        if n.endswith(" " + u) or n.endswith(u):
            n2 = n.replace(u, "").strip()
            return n2, normalize_unit(u)

    for u in ["kg", "ml", "g", "l", "L"]:
        if re.search(rf'\b{u}\b\s*$', n):
            n2 = re.sub(rf'\b{u}\b\s*$', '', n).strip()
            return n2, normalize_unit(u)

    return n, unit_code

# =========================
# 핵심 파서
# =========================
def parse_ingredient(text):
    raw = clean_text(text)

    # 설명/문장 컷
    if is_instruction_or_explain(raw):
        return None

    # 목록번호 제거 (1. 2. 3. 같은 거)
    if looks_like_list_index(raw):
        raw = re.sub(r'^\d{1,2}\s+', '', raw).strip()
        if is_instruction_or_explain(raw):
            return None

    # 1) 숫자 + 단위 추출
    num_match = re.search(r"(\d+\/\d+|\d+\.?\d*)\s*(kg|g|그램|ml|L|l|컵|큰술|작은술|개)?", raw)

    quantity = None
    unit_code = None

    if num_match:
        raw_q = num_match.group(1)
        raw_unit = num_match.group(2)
        quantity = convert_fraction(raw_q)
        unit_code = normalize_unit(raw_unit)
        raw = raw.replace(num_match.group(0), "").strip()

    # 2) 비정량 표현(note)
    approx_match = re.search(APPROX_PATTERN, raw)
    note = None
    if approx_match:
        note = approx_match.group(0)
        raw = raw.replace(note, "").strip()

    # 3) 문자 정리 (한글/영문/공백만)
    raw = re.sub(r"[^가-힣a-zA-Z\s]", "", raw).strip()
    if len(raw) <= 1:
        return None

    # 4) 끝 조사 제거
    raw = re.sub(r"(을|를|에|에서|으로|까지)$", "", raw).strip()
    if raw == "":
        return None

    # 5) 영어-only 제거
    if is_english_only(raw):
        return None

    # 6) 재료명 끝 단위 이동
    ingredient_name, unit_code = move_trailing_unit_to_unitcode(raw, unit_code)
    ingredient_name = ingredient_name.strip()

    if ingredient_name == "" or len(ingredient_name) <= 1:
        return None

    # 최종으로 한번 더 설명문 컷(재료명으로 남았어도 문장이면 제거)
    if is_instruction_or_explain(ingredient_name):
        return None

    base_quantity = convert_to_base(quantity, unit_code)

    return ingredient_name, quantity, unit_code, base_quantity, note

# =========================
# 실행
# =========================
encoding_type = detect_encoding(INPUT_FILE)
print("감지된 인코딩:", encoding_type)

df = pd.read_csv(INPUT_FILE, encoding=encoding_type)

parsed_rows = []
for _, row in df.iterrows():
    recipe_id = row.get("recipe_id")
    raw_ingredients = row.get("재료")

    if pd.isna(raw_ingredients):
        continue

    for item in str(raw_ingredients).split(";"):
        result = parse_ingredient(item)
        if result is None:
            continue

        name, quantity, unit_code, base_quantity, note = result
        parsed_rows.append({
            "recipe_external_id": recipe_id,
            "ingredient_name_norm": name,
            "quantity": quantity,
            "unit_code": unit_code,
            "base_quantity": base_quantity,
            "note": note
        })

parsed_df = pd.DataFrame(parsed_rows).drop_duplicates()
parsed_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("✅ v3 정제 완료")
print("총 행 수:", len(parsed_df))
print("저장 위치:", OUTPUT_FILE)
print(parsed_df.head(10))


감지된 인코딩: UTF-8-SIG
✅ v3 정제 완료
총 행 수: 4093
저장 위치: C:/Users/alstj/Desktop/백종원레시피정제_final_stable_v3.csv
  recipe_external_id ingredient_name_norm  quantity unit_code  base_quantity  \
0        wDXCxlcr3hY                돼지 목살     300.0         g          300.0   
1        wDXCxlcr3hY            꽃소금과 후춧가루       NaN      None            NaN   
2        wDXCxlcr3hY           파프리카 각 씩 청      50.0         g           50.0   
3        wDXCxlcr3hY                   감자     150.0         g          150.0   
4        wDXCxlcr3hY                   당근      70.0         g           70.0   
5        wDXCxlcr3hY                   양파     200.0         g          200.0   
6        wDXCxlcr3hY                  식용유      30.0        ml           30.0   
7        wDXCxlcr3hY                 고형카레      60.0         g           60.0   
8        thXIVUt9PBU                 채소믹스       NaN      None            NaN   
9        thXIVUt9PBU                   양파       2.0        kg         2000.0   

   note  
0  None 